In [71]:
import os
import numpy as np
from dataclasses import dataclass
from pathlib import Path
import pandas as pd 
from sentence_transformers import SentenceTransformer
import faiss
import pickle
from ollama import Client

In [72]:
@dataclass(frozen=True)
class AIRecommendationConfig:
    root_dir : Path 
    source_path : str 
    local_datafile : str

In [73]:
pwd

'c:\\Users\\RSR\\PYTHON\\ServicenowGamification\\src'

In [74]:
os.chdir('c:\\Users\\RSR\\PYTHON\\ServicenowGamification\\src')

In [75]:
from ServicenowGamification.Utils.Common import *
from ServicenowGamification.Constant import * 
from ServicenowGamification.Entity.Config_Entity import * 
from ServicenowGamification.Components.Data_Sns_LLM_Embendings import CreateEmbedings,PromptBuilder


In [76]:
os.chdir('c:\\Users\\RSR\\PYTHON\\ServicenowGamification')

In [77]:
class ConfigurationManager:
    def __init__(self,
                 config_path = CONFIG_FILE_PATH,
                 params_path =PARAMS_FILE_PATH
                 ):
        
        self.config=Read_Yaml(Path(config_path))
        self.params=Read_Yaml(Path(params_path))

        self.AI_Recommendation=self.config.AI_Recommendations
        self.Create_Embeddings=self.config.Create_Embeddings

    def get_create_embendings_config(self) -> CreateEmbendingsConfig:
            Config=self.Create_Embeddings
            create_directories([Config.root_dir])
            
            Create_Embeddings_Config=CreateEmbendingsConfig(
                root_dir=Config.root_dir,
                source_path=Config.source_path,
                faiss_index=Config.faiss_index,
                provider=Config.provider,
                model=Config.model, 
                base_url=Config.base_url, 
                temperature =Config.temperature,
                top_p =Config.top_p,
                max_tokens =Config.max_tokens, 
                timeout =Config.timeout,
                embedding_model= Config.embedding_model,
                load_faiss_index = Config.load_faiss_index,
                documents_path =Config.documents_path ,
                top_k =Config.top_k ,
                system_prompt=Config.system_prompt,
            )
            return Create_Embeddings_Config
    
    def get_ai_recommendation_config(self) -> AIRecommendationConfig:
        Config=self.AI_Recommendation
        create_directories([Config.root_dir])
        
        ai_recommendation_config=AIRecommendationConfig(
            root_dir=Config.root_dir,
            source_path=Config.source_path,           
            local_datafile=Config.local_datafile
        )
        return ai_recommendation_config      



In [78]:
import pandas as pd

In [ ]:
import pandas as pd

import os
import pickle
import faiss
import numpy as np

from sentence_transformers import SentenceTransformer


class FAISSSearch:

    def __init__(
            self,
            config:AIRecommendationConfig,
            faiss_index_path,
            documents_path,
            embedding_model="sentence-transformers/all-MiniLM-L6-v2"
            
            ):
        self.config=config
        self.faiss_index_path = faiss_index_path
        self.documents_path = documents_path
        self.embedding_model_name = embedding_model

        self.index = None
        self.documents = None
        self.model = None

    # ============================================================
    # Load FAISS Index
    # ============================================================

    def load_faiss_index(self):

        if not os.path.exists(self.faiss_index_path):

            raise FileNotFoundError(
                f"FAISS index not found:\n"
                f"{self.faiss_index_path}"
            )

        print("Loading FAISS index...")

        self.index = faiss.read_index(
            self.faiss_index_path
        )

        print(
            "FAISS index loaded successfully."
        )

        print(
            "Total vectors:",
            self.index.ntotal
        )

    # ============================================================
    # Load Documents
    # ============================================================

    def load_documents(self):

        if not os.path.exists(self.documents_path):

            raise FileNotFoundError(
                f"Documents file not found:\n"
                f"{self.documents_path}"
            )

        print("Loading FAISS documents...")

        with open(
            self.documents_path,
            "rb"
        ) as f:

            self.documents = pickle.load(f)

        print(
            "Total documents:",
            len(self.documents)
        )

    # ============================================================
    # Load Embedding Model
    # ============================================================

    def load_embedding_model(self):

        print(
            "Loading embedding model..."
        )

        self.model = SentenceTransformer(
            self.embedding_model_name
        )

        print(
            "Embedding model loaded."
        )

    # ============================================================
    # Create Query Embedding
    # ============================================================

    def create_query_embedding(
        self,
        query
    ):

        if self.model is None:

            self.load_embedding_model()

        embedding = self.model.encode(
            [query],
            convert_to_numpy=True
        )

        embedding = embedding.astype(
            "float32"
        )

        return embedding

    # ============================================================
    # SEARCH
    # ============================================================

    def search(
        self,
        query,
        top_k=5
    ):

        print("=" * 70)
        print("SEARCHING RELEVANT INCIDENTS")
        print("=" * 70)

        # --------------------------------------------------------
        # Load resources
        # --------------------------------------------------------

        if self.index is None:
            self.load_faiss_index()

        if self.documents is None:
            self.load_documents()

        if self.model is None:
            self.load_embedding_model()

        # --------------------------------------------------------
        # Validate
        # --------------------------------------------------------

        if self.index.ntotal == 0:

            print(
                "FAISS index contains 0 vectors."
            )

            return []

        if len(self.documents) == 0:

            print(
                "No documents available."
            )

            return []

        # --------------------------------------------------------
        # Make sure K is valid
        # --------------------------------------------------------

        top_k = min(
            int(top_k),
            self.index.ntotal,
            len(self.documents)
        )

        # --------------------------------------------------------
        # Query embedding
        # --------------------------------------------------------

        query_embedding = (
            self.create_query_embedding(query)
        )

        # --------------------------------------------------------
        # FAISS search
        # --------------------------------------------------------

        distances, indices = self.index.search(
            query_embedding,
            top_k
        )

        # --------------------------------------------------------
        # Results
        # --------------------------------------------------------

        results = []

        for rank, (
            index_id,
            distance
        ) in enumerate(
            zip(
                indices[0],
                distances[0]
            ),
            start=1
        ):

            if index_id == -1:
                continue

            document = self.documents[
                int(index_id)
            ]

            results.append(
                {
                    "rank": rank,
                    "index": int(index_id),
                    "distance": float(distance),
                    "document": document
                }
            )

        # --------------------------------------------------------
        # Print
        # --------------------------------------------------------

        print(
            "\nQuery:"
        )

        print(query)

        print(
            "\nRetrieved incidents:",
            len(results)
        )

        for result in results:

            print("\n" + "-" * 70)

            print(
                "Rank:",
                result["rank"]
            )

            print(
                "Index:",
                result["index"]
            )

            print(
                "Distance:",
                round(
                    result["distance"],
                    4
                )
            )

            print(
                "Document:"
            )

            print(
                result["document"]
            )

        print("=" * 70)

        return results


class AIRecommendation:

    def __init__(
        self,
        config,
        llm,
        prompt_builder,
        faiss_search=None
    ):

        self.config = config

        # Reuse existing objects
        self.llm = llm
        self.prompt = prompt_builder

        # FAISS search object
        self.faiss_search = faiss_search


    # ============================================================
    # Generate Recommendation - Single Incident
    # ============================================================

    def generate_recommendation(
        self,
        incident_data,
        sla_data=None,
        workload_data=None,
        performance_data=None,
        gamification_data=None,
        retrieved_incidents=None
    ):

        print("=" * 80)
        print("STEP 41 - AI RECOMMENDATION")
        print("=" * 80)

        # --------------------------------------------------------
        # Build Prompt
        # --------------------------------------------------------

        prompt = self.prompt.build_prompt(
            task="recommendation",
            incident_data=incident_data,
            sla_data=sla_data,
            workload_data=workload_data,
            performance_data=performance_data,
            gamification_data=gamification_data,
            retrieved_incidents=retrieved_incidents
        )

        print("Prompt created successfully.")

        # --------------------------------------------------------
        # Generate LLM Response
        # --------------------------------------------------------

        response = self.llm.generate_response(prompt)

        print("\n" + "=" * 80)
        print("AI RECOMMENDATION")
        print("=" * 80)

        print(response)

        return response


    # ============================================================
    # Generate Recommendations - Multiple Incidents
    # ============================================================

    def generate_recommendations(self):

        print("=" * 80)
        print("GENERATING AI RECOMMENDATIONS")
        print("=" * 80)

        # ------------------------------------------------------------
        # Load CSV
        # ------------------------------------------------------------

        source_path = str( self.config.source_path)

        print("Source:",source_path)

        if not os.path.exists(source_path):

            raise FileNotFoundError(f"Source file not found:\n{source_path}")

        df = pd.read_csv(source_path)

        print("Total incidents:",len(df))

        # ------------------------------------------------------------
        # Connect Ollama ONCE
        # ------------------------------------------------------------

        self.llm.connect()

        self.llm.test_connection()

        # ------------------------------------------------------------
        # Load FAISS ONCE
        # ------------------------------------------------------------

        self.search()
        # self.search.load_documents()
        # self.search.load_embedding_model()

        # ------------------------------------------------------------
        # Recommendation column
        # ------------------------------------------------------------

        if "ai_recommendation" not in df.columns:
            df["ai_recommendation"] = ""

        # ============================================================
        # PROCESS EACH INCIDENT
        # ============================================================

        for index, row in df.iterrows():
            try:

                print("\n")
                print("=" * 80)

                print( f"Processing {index + 1}/{len(df)}")

                print("Incident:",row.get("number", ""))

                print("=" * 80)

                # ----------------------------------------------------
                # Incident data
                # ----------------------------------------------------

                incident_data = (row.to_dict())

                # ----------------------------------------------------
                # SLA
                # ----------------------------------------------------

                sla_data = {

                    "sla_status":
                        row.get(
                            "sla_status",
                            "Not Available"
                        ),

                    "sla_prediction":
                        row.get(
                            "sla_prediction",
                            "Not Available"
                        ),

                    "sla_confidence":
                        row.get(
                            "sla_confidence",
                            "Not Available"
                        ),

                    "sla_percentage":
                        row.get(
                            "sla_percentage",
                            "Not Available"
                        )
                }

                # ----------------------------------------------------
                # Workload
                # ----------------------------------------------------

                workload_data = {

                    "workload":
                        row.get(
                            "workload",
                            "Not Available"
                        ),

                    "workload_level":
                        row.get(
                            "workload_level",
                            "Not Available"
                        ),

                    "workload_confidence":
                        row.get(
                            "workload_confidence",
                            "Not Available"
                        )
                }

                # ----------------------------------------------------
                # Performance
                # ----------------------------------------------------

                performance_data = {

                    "assigned_to":
                        row.get(
                            "assigned_to",
                            "Not Available"
                        ),

                    "performance_prediction":
                        row.get(
                            "performance_prediction",
                            "Not Available"
                        ),

                    "performance_category":
                        row.get(
                            "performance_category",
                            "Not Available"
                        ),

                    "performance_confidence":
                        row.get(
                            "performance_confidence",
                            "Not Available"
                        ),

                    "quality_score":
                        row.get(
                            "quality_score",
                            "Not Available"
                        ),

                    "skill_score":
                        row.get(
                            "skill_score",
                            "Not Available"
                        )
                }

                # ----------------------------------------------------
                # Gamification
                # ----------------------------------------------------

                gamification_data = {

                    "total_points":
                        row.get(
                            "total_points",
                            "Not Available"
                        ),

                    "points":
                        row.get(
                            "points",
                            "Not Available"
                        ),

                    "badges":
                        row.get(
                            "badges",
                            "Not Available"
                        ),

                    "rewards":
                        row.get(
                            "rewards",
                            "Not Available"
                        ),

                    "achievements":
                        row.get(
                            "achievements",
                            "Not Available"
                        ),

                    "achievement":
                        row.get(
                            "achievement",
                            "Not Available"
                        ),

                    "level":
                        row.get(
                            "level",
                            "Not Available"
                        ),

                    "leaderboard_rank":
                        row.get(
                            "leaderboard_rank",
                            "Not Available"
                        ),

                    "monthly_rank":
                        row.get(
                            "monthly_rank",
                            "Not Available"
                        )
                }

                # ====================================================
                # CREATE FAISS QUERY
                # ====================================================

                query = self.create_search_query(
                    incident_data
                )

                print(
                    "\nSearching similar incidents..."
                )

                # ====================================================
                # FAISS SEARCH
                # ====================================================

                retrieved_incidents = (
                    self.faiss_search.search(
                        query=query,
                        top_k=self.config.top_k
                    )
                )

                print(
                    "Similar incidents found:",
                    len(retrieved_incidents)
                )

                # ====================================================
                # BUILD PROMPT
                # ====================================================

                prompt = self.prompt.build_prompt(

                    task="recommendation",

                    incident_data=incident_data,

                    sla_data=sla_data,

                    workload_data=workload_data,

                    performance_data=performance_data,

                    gamification_data=gamification_data,

                    retrieved_incidents=retrieved_incidents
                )

                # ====================================================
                # GENERATE RESPONSE
                # ====================================================

                response = (
                    self.llm.generate_response(
                        prompt
                    )
                )

                # ====================================================
                # SAVE
                # ====================================================

                df.loc[
                    index,
                    "ai_recommendation"
                ] = response

                print(
                    "Recommendation generated successfully."
                )

            except Exception as e:

                print(
                    f"AI recommendation failed for "
                    f"{row.get('number', index)}: {e}"
                )

                df.loc[
                    index,
                    "ai_recommendation"
                ] = (
                    "AI recommendation generation failed: "
                    + str(e)
                )


In [82]:
config_manager = ConfigurationManager()

config = (
    config_manager
    .get_create_embendings_config()
)


# ------------------------------------------------------------
# LLM
# ------------------------------------------------------------

llm = CreateEmbedings(config)
llm.connect()
llm.test_connection()


# ------------------------------------------------------------
# Prompt Builder
# ------------------------------------------------------------

prompt_builder = PromptBuilder(
    config
)


# ------------------------------------------------------------
# FAISS Search
# ------------------------------------------------------------

# faiss_search = FAISSSearch(
#     config
# )

faiss_search=[]
# ------------------------------------------------------------
# AI Recommendation
# ------------------------------------------------------------

recommendation = AIRecommendation(

    config=config,

    llm=llm,

    prompt_builder=prompt_builder,

    faiss_search=faiss_search
)


# ------------------------------------------------------------
# Generate recommendations
# ------------------------------------------------------------

df = recommendation.generate_recommendations()


# ------------------------------------------------------------
# Display result
# ------------------------------------------------------------

print(
    df[
        [
            "number",
            "assigned_to",
            "ai_recommendation"
        ]
    ]
)

Reading YAML : Config\Config.yaml
[2026-08-10 18:44:19,305: INFO: Common: YAML file loaded successfully : Config\Config.yaml]


Reading YAML : params.yaml
[2026-08-10 18:44:19,435: INFO: Common: YAML file loaded successfully : params.yaml]
[2026-08-10 18:44:19,457: INFO: Common: Created directory : Artifacts/LLMEmbenddings]
Reading YAML : params.yaml
[2026-08-10 18:44:19,598: INFO: Common: YAML file loaded successfully : params.yaml]
Connecting to Ollama Server
Host  : http://127.0.0.1:11434
Model : llama3.2
Ollama Client Created Successfully

Checking Ollama Server...
[2026-08-10 18:44:19,865: INFO: _client: HTTP Request: GET http://127.0.0.1:11434/api/tags "HTTP/1.1 200 OK"]
Connection Successful

Available Models:
 - llama3.2:latest
GENERATING AI RECOMMENDATIONS
Source: Artifacts/AgentPerformance/AgentPerformancePrediction.csv
Total incidents: 63
Connecting to Ollama Server
Host  : http://127.0.0.1:11434
Model : llama3.2
Ollama Client Created Successfully

Checking Ollama Server...
[2026-08-10 18:44:20,049: INFO: _client: HTTP Request: GET http://127.0.0.1:11434/api/tags "HTTP/1.1 200 OK"]
Connection Success

AttributeError: 'AIRecommendation' object has no attribute 'search'

In [68]:
os.chdir('c:\\Users\\RSR\\PYTHON\\ServicenowGamification\\src')

In [ ]:

config_manager = ConfigurationManager()

    config = (
        config_manager
        .get_ai_recommendation_config()
    )

    # ------------------------------------------------------------
    # AI Recommendation
    # ------------------------------------------------------------

    recommendation = AIRecommendation(
        config
    )

    # ------------------------------------------------------------
    # Generate
    # ------------------------------------------------------------

    df = (
        recommendation
        .generate_recommendations()
    )

    # ------------------------------------------------------------
    # Display
    # ------------------------------------------------------------

    print(
        df[
            [
                "number",
                "assigned_to",
                "ai_recommendation"
            ]
        ]
    )

ModuleNotFoundError: No module named 'ServicenowGamification.Components.AIRecommendation'